# 📉 Müşteri Kaybı (Churn) Analizi — Keşifçi Veri Analizi (EDA)

Bu notebook, churn tahmin projesinin **veri keşfi** adımını anlatır. Amaç: modeli kurmadan önce
"veride ne var, müşteriler neden ayrılıyor?" sorusunu yanıtlamak.

**Veri:** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — ~7.000 telekom müşterisi.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Boyut:", df.shape)
df.head()

## 1. Veriye İlk Bakış

Sütun tiplerine ve eksik değerlere bakalım. `TotalCharges` sayısal olması gerekirken metin (object)
olarak gelmiş — bu, temizlenmesi gereken klasik bir sorundur.

In [ ]:
df.info()

In [ ]:
# TotalCharges'i sayisala cevir; bos degerler NaN olur
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges'taki bos deger sayisi:", df["TotalCharges"].isna().sum())

## 2. Hedef Değişken: Kaç Müşteri Ayrılıyor?

Önce churn oranına bakalım. Veri **dengesiz** mi? (Genelde ayrılanlar azınlıktadır — bu, model
seçiminde ve metriklerde önemlidir.)

In [ ]:
churn_rate = df["Churn"].value_counts(normalize=True) * 100
print(churn_rate.round(1))

plt.figure(figsize=(5, 4))
sns.countplot(x="Churn", data=df, palette="Set2")
plt.title("Müşteri Kaybı Dağılımı")
plt.show()

**Bulgu:** Müşterilerin yaklaşık **%26'sı** ayrılmış. Veri dengesiz; bu yüzden modelde
`class_weight="balanced"` kullanacağız ve sadece *accuracy*'ye değil **recall** ve **ROC-AUC**'ye bakacağız.

## 3. Sözleşme Tipi ve Churn

İş açısından en kritik grafiklerden biri: sözleşme tipi churn'ü nasıl etkiliyor?

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x="Contract", hue="Churn", data=df, palette="Set2")
plt.title("Sözleşme Tipine Göre Müşteri Kaybı")
plt.xticks(rotation=10)
plt.show()

# Sozlesme tipine gore churn orani
print((df.groupby("Contract")["Churn"]
         .apply(lambda s: (s == "Yes").mean() * 100).round(1)))

**Bulgu:** **Aya-dayalı (Month-to-month)** sözleşmeli müşterilerde churn dramatik biçimde yüksek.
Uzun vadeli (1-2 yıllık) sözleşmeler müşteriyi elde tutuyor. → Şirkete somut öneri:
müşterileri uzun vadeli sözleşmelere teşvik et.

## 4. Müşteri Süresi (tenure) ve Aylık Ücret

Yeni müşteriler mi yoksa eski müşteriler mi daha çok ayrılıyor? Ücret churn'ü etkiliyor mu?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", bins=30, ax=axes[0], palette="Set2")
axes[0].set_title("Müşteri Süresi (ay) ve Churn")
sns.boxplot(x="Churn", y="MonthlyCharges", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("Aylık Ücret ve Churn")
plt.tight_layout()
plt.show()

**Bulgu:** Churn çoğunlukla **ilk aylarda** (düşük tenure) yaşanıyor; müşteri ne kadar uzun kalırsa
ayrılma ihtimali o kadar düşüyor. Ayrıca **yüksek aylık ücret** ödeyenler daha çok ayrılıyor.

## 5. Sayısal Değişkenler Arası İlişki (korelasyon)

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
plt.figure(figsize=(5, 4))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Sayısal Değişken Korelasyonları")
plt.show()

## 🧭 Özet Bulgular

1. **~%26 churn** — veri dengesiz, doğru metrik seçimi önemli.
2. **Aya-dayalı sözleşmeler** en riskli grup.
3. **Düşük tenure (yeni müşteriler)** çok daha fazla ayrılıyor.
4. **Yüksek aylık ücret** churn'ü artırıyor.

Bu bulgular, `train_model.py`'de kurduğumuz modelin **feature importance** ve **SHAP** sonuçlarıyla
birebir örtüşüyor — yani model, veride gözle gördüğümüz örüntüleri öğrenmiş.

➡️ Sonraki adım: `train_model.py` ile modeli eğitmek ve `streamlit run app.py` ile arayüzü açmak.